# BITS Pilani M.Tech AIML – Machine Learning Assignment 2

### Assignment workflow
1. Obtain and validate a public classification dataset (minimum 500 rows and 12 independent features).
2. Explore the dataset and prepare the data.
3. Train Logistic Regression, Decision Tree, KNN, Naive Bayes, and SVM on the same train/test split.
4. Evaluate Accuracy, AUC, Precision, Recall, F1 Score, and MCC.
5. Save models and supporting artifacts.
6. Generate a Streamlit application that loads saved models, accepts a labeled test CSV, and compares models without retraining.
7. Prepare README and deployment instructions.


## Dry Bean Multi-Class Classification

**Dataset:** UCI Dry Bean Dataset  
**Problem type:** Supervised multi-class classification  
**Target:** `Class` (seven dry bean varieties)



In [1]:
%pip install ucimlrepo pandas numpy matplotlib seaborn scikit-learn joblib streamlit

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import json
import warnings
from pathlib import Path

In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from ucimlrepo import fetch_ucirepo

In [8]:
warnings.filterwarnings("ignore")
RANDOM_STATE = 42
TEST_SIZE = 0.20

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
MODEL_DIR = PROJECT_DIR / "models"
REPORT_DIR = PROJECT_DIR / "reports"

for directory in (DATA_DIR, MODEL_DIR, REPORT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR.resolve())

Project directory: F:\GitWS\ml_classification_comparison


In [9]:
# Download the dataset from the UCI repository.
dry_bean = fetch_ucirepo(id=602)

X_raw = dry_bean.data.features.copy()
y_raw = dry_bean.data.targets.copy()

if isinstance(y_raw, pd.DataFrame):
    if y_raw.shape[1] != 1:
        raise ValueError(f"Expected one target column, received: {list(y_raw.columns)}")
    y_raw = y_raw.iloc[:, 0]

target_column = "Class"
X_raw.columns = [str(c).strip() for c in X_raw.columns]
y_raw = y_raw.astype(str).rename(target_column)

df = X_raw.copy()
df[target_column] = y_raw

# Convert feature columns to numeric, raising an error if unexpected text is present.
for col in X_raw.columns:
    df[col] = pd.to_numeric(df[col], errors="raise")

df.to_csv(DATA_DIR / "dry_bean.csv", index=False)

print("Dataset downloaded and saved.")
print("Shape:", df.shape)
print("Input features:", len(X_raw.columns))
print("Target classes:", sorted(df[target_column].unique()))
display(df.head())

Dataset downloaded and saved.
Shape: (13611, 17)
Input features: 16
Target classes: ['BARBUNYA', 'BOMBAY', 'CALI', 'DERMASON', 'HOROZ', 'SEKER', 'SIRA']


,Area,Perimeter,MajorAxisLength,MinorAxisLength,AspectRatio,Eccentricity,ConvexArea,EquivDiameter,Extent,Solidity,Roundness,Compactness,ShapeFactor1,ShapeFactor2,ShapeFactor3,ShapeFactor4,Class
0,28395,610.291,208.178117,173.888747,1.197191,0.549812,28715,190.141097,0.763923,0.988856,0.958027,0.913358,0.007332,0.003147,0.834222,0.998724,SEKER
1,28734,638.018,200.524796,182.734419,1.097356,0.411785,29172,191.272751,0.783968,0.984986,0.887034,0.953861,0.006979,0.003564,0.909851,0.998430,SEKER
2,29380,624.110,212.826130,175.931143,1.209713,0.562727,29690,193.410904,0.778113,0.989559,0.947849,0.908774,0.007244,0.003048,0.825871,0.999066,SEKER
3,30008,645.884,210.557999,182.516516,1.153638,0.498616,30724,195.467062,0.782681,0.976696,0.903936,0.928329,0.007017,0.003215,0.861794,0.994199,SEKER
4,30140,620.134,201.847882,190.279279,1.060798,0.333680,30417,195.896503,0.773098,0.990893,0.984877,0.970516,0.006697,0.003665,0.941900,0.999166,SEKER
